In [1]:
import os
import pandas as pd

base_input = "/kaggle/input/plant-pathology-2021-fgvc8"
output_dir = "./"

test_dir = os.path.join(base_input, "test_images")
model_dir = "/kaggle/input/dense-e1/epoch-1"  # may not exist in this environment
image_dims = (300, 300, 3)

train_path = os.path.join(base_input, "train.csv")
data_set = pd.read_csv(train_path)
df_labels = data_set["labels"]
one_hot = df_labels.str.get_dummies(sep=" ")
dataset_labels = one_hot.columns.tolist()

label_counts = one_hot.sum().sort_values(ascending=False)

# Use only the most frequent non‑complex label as baseline (usually “healthy”)
top_n = 1
baseline_labels = [lbl for lbl in label_counts.index if lbl != "complex"][:top_n]
baseline_str = " ".join(baseline_labels)




In [2]:
use_model = os.path.isdir(model_dir)

if use_model:
    try:
        import tensorflow as tf

        model_layer = tf.keras.layers.TFSMLayer(
            model_dir, call_endpoint="serving_default"
        )
        images_path_list = sorted(
            [f for f in os.listdir(test_dir) if f.lower().endswith(".jpg")]
        )
        values = []
        # Raise confidence threshold to improve precision
        thresh = 0.5
        # Limit to top‑2 predictions per image
        max_labels = 2
        for img_name in images_path_list:
            img_path = os.path.join(test_dir, img_name)
            raw_img = tf.io.read_file(img_path)
            img = tf.io.decode_image(
                contents=raw_img, channels=3, dtype=tf.dtypes.float32
            )
            img = tf.image.resize(img, [image_dims[0], image_dims[1]])
            img = tf.expand_dims(img, axis=0)  # add batch dimension
            img = img * 255.0  # match training scale if needed

            preds = model_layer(img)  # (1, num_classes)
            preds = preds.numpy()[0]  # convert to NumPy array

            index_values = [i for i, v in enumerate(preds) if v > thresh]

            if index_values:
                index_values = sorted(
                    index_values, key=lambda i: preds[i], reverse=True
                )[:max_labels]
                labels_str = " ".join(dataset_labels[i] for i in index_values)
            else:
                labels_str = baseline_str

            values.append([img_name, labels_str])
    except Exception as e:
        print(f"Model loading failed ({e}); falling back to baseline predictions.")
        use_model = False

if not use_model:
    images_path_list = sorted(
        [f for f in os.listdir(test_dir) if f.lower().endswith(".jpg")]
    )
    values = [[img_name, baseline_str] for img_name in images_path_list]

submission_df = pd.DataFrame(values, columns=["image", "labels"])
submission_path = os.path.join(output_dir, "submission.csv")
submission_df.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")

Submission written to ./submission.csv
